# PaceAlgo ML — Notebook 12: Model Battery

Vergleich von 4 Modellarchitekturen auf der **besten Feature-Konfiguration aus NB 11** (`phase1_best_config.json`):

1. **LightGBM** (Pine-budget: 30 trees, depth 3) — bisheriges baseline
2. **XGBoost** (gleiche depth + trees) — kann marginal andere Splits finden
3. **CatBoost** (gleiche) — bessere categorical handling, oft robusterer Default
4. **Voting Ensemble** — Mittelwert der 3 calibrated probabilities

**Strikte Methodik:**
- Identische Train/Val/Test-Splits (Walk-Forward)
- Identische Tier-Cutoffs aus VAL-Set abgeleitet
- Pro Modell × Tier: PF / WR / Expectancy / Stability-CV
- Pine-Export-Eignung berücksichtigt (LGBM/XGB exportierbar; CatBoost komplex)

**Entscheidungslogik:**
- Bestes Modell muss `Premium-Tier-PF > NB-11-Bestwert + 0.05` haben um den Wechsel zu rechtfertigen
- Sonst bleibt LightGBM (einfachster Pine-Export)

## 1. Setup

In [ ]:
import sys, os
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/pace-algo'
    DATA_PROCESSED = Path('/content/processed')
    DATA_EXT = Path('/content/processed_v2')
    ARTIFACTS = Path(PROJECT_ROOT) / 'artifacts'
    REPORTS_DIR = ARTIFACTS / 'reports'
    MODELS_DIR = ARTIFACTS / 'models'
    os.chdir(PROJECT_ROOT)
    !rm -rf /tmp/pace-algo
    !git clone -q https://github.com/ecoNC/pace-algo.git /tmp/pace-algo
    !cp -rf /tmp/pace-algo/core/* {PROJECT_ROOT}/core/
    print('Core code updated from GitHub')
else:
    PROJECT_ROOT = os.path.abspath('..')
    os.chdir(PROJECT_ROOT)
    from core.config import DATA_PROCESSED, ARTIFACTS_REPORTS, ARTIFACTS_MODELS
    REPORTS_DIR = ARTIFACTS_REPORTS
    MODELS_DIR = ARTIFACTS_MODELS
    DATA_EXT = DATA_PROCESSED.parent / 'processed_v2'

MODELS_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, PROJECT_ROOT)
for mod in list(sys.modules.keys()):
    if mod.startswith('core'):
        del sys.modules[mod]

In [ ]:
!pip install -q lightgbm xgboost catboost 2>&1 | tail -1

import json
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score

from core.config import (
    FX_SYMBOLS, METAL_SYMBOLS, PRIMARY_TIMEFRAMES, DEV_HOLDOUT_SYMBOLS,
    TRAIN_END, VAL_END,
)
from core.train import walk_forward_split, binary_label_for_long

R_VALUE = 1.5
WIN_R = 1.5
LOSS_R = 1.0
print('Libraries loaded:')
print(f'  LightGBM: {lgb.__version__}')
print(f'  XGBoost:  {xgb.__version__}')
import catboost
print(f'  CatBoost: {catboost.__version__}')

## 2. Load Best Config from NB 11

In [ ]:
config_path = REPORTS_DIR / 'phase1_best_config.json'
if not config_path.exists():
    print(f'ERR: {config_path} missing. Run NB 11 first.')
else:
    with open(config_path) as f:
        best_config = json.load(f)
    print('Loaded best config from NB 11:')
    print(f'  experiment_name: {best_config["experiment_name"]}')
    print(f'  asset_scope:     {best_config["asset_scope"]}')
    print(f'  n_features:      {best_config["n_features"]}')
    print(f'  PF (NB11):       {best_config["premium_pf_oos"]:.4f}')
    print(f'  WR (NB11):       {best_config["premium_wr_oos"]*100:.1f}%')
    print(f'  ExpR (NB11):     {best_config["premium_expR_oos"]:+.4f}')
    print(f'  features: {best_config["feature_cols"][:5]} ... (+{len(best_config["feature_cols"])-5} more)')

## 3. Load Extended Dataset (same as NB 11)

In [ ]:
def load_ext(sym, tf):
    p = DATA_EXT / f'{sym}_{tf}_extended.parquet'
    return pd.read_parquet(p) if p.exists() else None

scope = best_config['asset_scope']
if scope == 'FX_only':
    symbols = FX_SYMBOLS
elif scope == 'Gold_only':
    symbols = METAL_SYMBOLS
else:
    symbols = FX_SYMBOLS + METAL_SYMBOLS

drop = set(DEV_HOLDOUT_SYMBOLS)
frames = []
for s in symbols:
    if s in drop: continue
    for tf in PRIMARY_TIMEFRAMES:
        d = load_ext(s, tf)
        if d is not None and not d.empty:
            frames.append(d)
df_full = pd.concat(frames, axis=0).sort_index() if frames else pd.DataFrame()
print(f'Stacked dataset for scope "{scope}": {df_full.shape}')

features = best_config['feature_cols']
available = [f for f in features if f in df_full.columns]
print(f'Available features: {len(available)} of {len(features)}')

df_c = df_full.dropna(subset=available + ['label'])
train_df, val_df, test_df = walk_forward_split(df_c, TRAIN_END, VAL_END)
print(f'train: {len(train_df):,}  val: {len(val_df):,}  test: {len(test_df):,}')

X_tr = train_df[available].values; y_tr = binary_label_for_long(train_df['label']).values
X_vl = val_df[available].values; y_vl = binary_label_for_long(val_df['label']).values
X_ts = test_df[available].values; y_ts = binary_label_for_long(test_df['label']).values

print(f'\nClass balance — train: {y_tr.mean():.3f}, val: {y_vl.mean():.3f}, test: {y_ts.mean():.3f}')

## 4. Train 3 Models with Equivalent Hyperparameters

In [ ]:
# Common hyperparams: 30 trees, depth 3, mild L2 reg
models = {}

print('Training LightGBM...')
lgb_params = {
    'objective': 'binary', 'metric': 'binary_logloss',
    'num_leaves': 7, 'max_depth': 3, 'min_data_in_leaf': 200,
    'learning_rate': 0.05, 'num_iterations': 30, 'lambda_l2': 0.5,
    'feature_fraction': 0.85, 'bagging_fraction': 0.85, 'bagging_freq': 5,
    'is_unbalance': True, 'verbose': -1, 'n_jobs': -1,
}
td = lgb.Dataset(X_tr, label=y_tr)
vd = lgb.Dataset(X_vl, label=y_vl, reference=td)
models['LightGBM'] = lgb.train(lgb_params, td, num_boost_round=30,
                                  valid_sets=[vd], valid_names=['val'],
                                  callbacks=[lgb.log_evaluation(period=0)])
print('  LightGBM done')

In [ ]:
print('Training XGBoost...')
pos_weight = (1 - y_tr.mean()) / y_tr.mean()  # equivalent to is_unbalance
xgb_clf = xgb.XGBClassifier(
    n_estimators=30,
    max_depth=3,
    learning_rate=0.05,
    reg_lambda=0.5,
    subsample=0.85,
    colsample_bytree=0.85,
    scale_pos_weight=pos_weight,
    use_label_encoder=False,
    eval_metric='logloss',
    n_jobs=-1,
    verbosity=0,
)
xgb_clf.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)], verbose=False)
models['XGBoost'] = xgb_clf
print('  XGBoost done')

In [ ]:
print('Training CatBoost...')
cat_clf = CatBoostClassifier(
    iterations=30,
    depth=3,
    learning_rate=0.05,
    l2_leaf_reg=3.0,
    rsm=0.85,
    auto_class_weights='Balanced',
    verbose=False,
    thread_count=-1,
)
cat_clf.fit(X_tr, y_tr, eval_set=(X_vl, y_vl), use_best_model=False)
models['CatBoost'] = cat_clf
print('  CatBoost done')

print('\nAll 3 base models trained.')

## 5. Predict + Voting Ensemble

In [ ]:
val_probas = {}
test_probas = {}

val_probas['LightGBM'] = models['LightGBM'].predict(X_vl)
test_probas['LightGBM'] = models['LightGBM'].predict(X_ts)

val_probas['XGBoost'] = models['XGBoost'].predict_proba(X_vl)[:, 1]
test_probas['XGBoost'] = models['XGBoost'].predict_proba(X_ts)[:, 1]

val_probas['CatBoost'] = models['CatBoost'].predict_proba(X_vl)[:, 1]
test_probas['CatBoost'] = models['CatBoost'].predict_proba(X_ts)[:, 1]

# Voting ensemble = average
val_probas['Voting'] = (val_probas['LightGBM'] + val_probas['XGBoost'] + val_probas['CatBoost']) / 3
test_probas['Voting'] = (test_probas['LightGBM'] + test_probas['XGBoost'] + test_probas['CatBoost']) / 3

print('All probas computed.')
for name in val_probas:
    print(f'  {name}: VAL range [{val_probas[name].min():.3f}, {val_probas[name].max():.3f}]')

## 6. Per-Model × Per-Tier Evaluation

In [ ]:
def tier_metrics(labels):
    wins = int((labels == 1).sum())
    losses = int((labels == -1).sum())
    neutrals = int((labels == 0).sum())
    total = wins + losses + neutrals
    pf = (wins * WIN_R) / (losses * LOSS_R) if losses > 0 else (float('inf') if wins > 0 else 0.0)
    wr = wins / (wins + losses) if (wins + losses) > 0 else 0.0
    expR = (wins * WIN_R - losses * LOSS_R) / total if total > 0 else 0.0
    return {'n': total, 'wins': wins, 'losses': losses, 'wr': wr, 'pf': pf, 'expR': expR}

rows = []
for model_name in ['LightGBM', 'XGBoost', 'CatBoost', 'Voting']:
    vp = val_probas[model_name]
    tp = test_probas[model_name]
    vs = np.sort(vp)[::-1]
    cutoff_std = float(vs[max(1, int(len(vs) * 0.10) - 1)])
    cutoff_high = float(vs[max(1, int(len(vs) * 0.03) - 1)])
    cutoff_prem = float(vs[max(1, int(len(vs) * 0.01) - 1)])
    # AUC on TEST for reference
    try:
        test_auc = float(roc_auc_score(y_ts, tp))
    except Exception:
        test_auc = float('nan')
    for tier_name, cutoff in [('Standard', cutoff_std), ('High', cutoff_high), ('Premium', cutoff_prem)]:
        mask = tp >= cutoff
        sub_labels = test_df['label'].iloc[mask.nonzero()[0]]
        m = tier_metrics(sub_labels)
        m['model'] = model_name
        m['tier'] = tier_name
        m['cutoff'] = cutoff
        m['auc'] = test_auc
        rows.append(m)

battery_df = pd.DataFrame(rows)

print('=== Profit Factor by model × tier ===')
display(battery_df.pivot_table(index='model', columns='tier', values='pf').round(4)[['Standard', 'High', 'Premium']])

print('\n=== Win Rate ===')
display(battery_df.pivot_table(index='model', columns='tier', values='wr').round(4)[['Standard', 'High', 'Premium']])

print('\n=== Expectancy per Trade ===')
display(battery_df.pivot_table(index='model', columns='tier', values='expR').round(4)[['Standard', 'High', 'Premium']])

print('\n=== Trade Count ===')
display(battery_df.pivot_table(index='model', columns='tier', values='n').round(0)[['Standard', 'High', 'Premium']])

print('\n=== TEST AUC per model ===')
display(battery_df.drop_duplicates('model')[['model', 'auc']].set_index('model').round(4))

## 7. Per-Year Stability per Model (Premium Tier)

In [ ]:
test_df_copy = test_df.copy()
test_df_copy['year'] = test_df_copy.index.year

stability_rows = []
for model_name in ['LightGBM', 'XGBoost', 'CatBoost', 'Voting']:
    vp = val_probas[model_name]
    tp = test_probas[model_name]
    vs = np.sort(vp)[::-1]
    cutoff_prem = float(vs[max(1, int(len(vs) * 0.01) - 1)])
    test_df_copy['proba'] = tp
    yearly = {}
    for year, sub in test_df_copy.groupby('year'):
        mask = sub['proba'] >= cutoff_prem
        sub_l = sub.loc[mask, 'label']
        if len(sub_l) < 30:
            continue
        m = tier_metrics(sub_l)
        yearly[int(year)] = m
    pf_values = [m['pf'] for m in yearly.values() if not np.isinf(m['pf']) and m['pf'] > 0]
    if len(pf_values) >= 2:
        cv = float(np.std(pf_values) / np.mean(pf_values))
    else:
        cv = float('nan')
    for year, m in yearly.items():
        stability_rows.append({'model': model_name, 'year': year, 'pf': m['pf'], 'wr': m['wr'], 'n': m['n']})
    print(f'{model_name}: years tested = {sorted(yearly.keys())}, stability CV = {cv:.3f}')
    print(f'  yearly PFs: {[(y, round(m["pf"], 3)) for y, m in yearly.items()]}')

stab_df = pd.DataFrame(stability_rows)
if not stab_df.empty:
    print('\nPremium PF per year per model:')
    display(stab_df.pivot_table(index='year', columns='model', values='pf').round(4))

## 8. Pine-Export-Eignung

In [ ]:
exportability = {
    'LightGBM': {'pine_ready': True,  'notes': 'Native tree-cascade, well-understood, our baseline'},
    'XGBoost':  {'pine_ready': True,  'notes': 'Similar tree structure, exportable with similar effort'},
    'CatBoost': {'pine_ready': False, 'notes': 'Oblivious trees + categorical embeddings — complex/impossible Pine export'},
    'Voting':   {'pine_ready': True,  'notes': 'Need to embed 3 models in Pine — ~3x line count, possible but bulky'},
}
exp_df = pd.DataFrame(exportability).T
print('Pine export feasibility per model:')
display(exp_df)

## 9. Decision Logic

In [ ]:
print('=' * 70)
print('MODEL BATTERY VERDICT')
print('=' * 70)

premium_pfs = battery_df[battery_df['tier'] == 'Premium'].set_index('model')['pf'].to_dict()
lgbm_pf = premium_pfs['LightGBM']
print(f'\nLightGBM baseline Premium PF: {lgbm_pf:.4f}')
print()
for name in ['XGBoost', 'CatBoost', 'Voting']:
    pf = premium_pfs[name]
    lift = pf - lgbm_pf
    pine_ready = exportability[name]['pine_ready']
    ready_str = 'Pine-ready' if pine_ready else 'Backend-only'
    print(f'  {name:<10s} PF {pf:.4f}  lift {lift:+.4f}  [{ready_str}]')

best_name = max(premium_pfs, key=premium_pfs.get)
best_pf = premium_pfs[best_name]
best_lift = best_pf - lgbm_pf
best_pine = exportability[best_name]['pine_ready']

print('\n' + '-' * 70)
print(f'Best model: {best_name} (Premium PF {best_pf:.4f}, lift {best_lift:+.4f})')
if best_name == 'LightGBM' or best_lift < 0.05:
    print('-> LightGBM remains the choice (lift too small to justify complexity)')
elif best_pine:
    print(f'-> {best_name} wins AND is Pine-exportable')
else:
    print(f'-> {best_name} wins but is NOT Pine-exportable')
    print(f'   Decision tree:')
    print(f'   - If V1 must be Pine-only: use 2nd-best Pine-ready model')
    print(f'   - If accept Backend-V1: use {best_name} via webhook → TradingView')

print('=' * 70)

---

Schick mir nach dem Run:
- Section 6 (PF/WR/Expectancy pro Modell × Tier + AUC)
- Section 7 (Per-Year Stability Tabelle)
- Section 8 (Pine-Export-Eignung)
- Section 9 (Verdict)

Damit treffen wir die finale Architekturentscheidung vor V1.